In [ ]:
# --- bootstrap: anchor to the repository root, wherever this notebook was opened from ---
# Notebooks live two levels deep under notebooks/, so the cwd-relative path logic below needs the
# root established first. Keyed on pytest.ini, which is not tied to any folder-naming decision.
import os
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"repo root: {_root}")

# RavenPack Macro News Extraction — Data Collection

**Academic research only — not investment advice.** Output of this project is a comparative evaluation of LLM-derived sentiment vs. market features for SP500 sector ETFs; nothing here is a trading signal.

This notebook is the data-engineering (silver/gold) counterpart to `Basic_EDA_Analysis.ipynb`: it re-runs the same RavenPack filtering logic but as a clean, reusable extraction pipeline rather than an exploratory notebook. It produces two outputs:

1. **Silver — core filtered event table** (`data/raw/ravenpack_core_events_<START>_<END>.csv`): row-level RavenPack macro events, 2020–2025, filtered to rank-1 non-blog sources and relevance/event_relevance ≥ 90, unioned across years into one table.
2. **Gold — daily summary table** (`news_daily_df`, saved to `data/news_daily_df.csv`): one row per NYSE trading session, aggregating the silver table with the after-hours (4:00 PM ET) lookahead-bias cutoff already applied.

**Licensing note:** WRDS RavenPack is a licensed academic dataset. The silver output is row-level and is written to `data/raw/`, which is `.gitignore`'d — it must never be committed. Only the aggregated gold table (`news_daily_df.csv`) is safe to commit.

In [1]:
%pip install -q wrds psycopg2-binary pandas numpy pyarrow pandas_market_calendars

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data").is_dir():
    REPO_ROOT = PROJECT_ROOT
    NOTEBOOK_DIR = REPO_ROOT / "data"
elif PROJECT_ROOT.name == "data" and (PROJECT_ROOT.parent / "data").is_dir():
    REPO_ROOT = PROJECT_ROOT.parent
    NOTEBOOK_DIR = PROJECT_ROOT
else:
    raise FileNotFoundError("Run this notebook from the repository root or data/.")
RAW_DIR = NOTEBOOK_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_DATE = "2015-01-01"
END_DATE = "2026-12-31"
YEARS = range(2015, 2027)

RELEVANCE_MIN = 90
EVENT_RELEVANCE_MIN = 90
MARKET_CLOSE_ET = "16:00:00"

CORE_EVENTS_CSV = RAW_DIR / f"ravenpack_core_events_{START_DATE[:4]}_{END_DATE[:4]}.csv"
NEWS_DAILY_CSV = NOTEBOOK_DIR / "news_daily_df.csv"

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Silver output (gitignored, raw/):  {CORE_EVENTS_CSV}")
print(f"Gold output (committed, aggregated): {NEWS_DAILY_CSV}")

Date range: 2020-01-01 to 2026-12-31
Silver output (gitignored, raw/):  C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\ravenpack_core_events_2020_2026.csv
Gold output (committed, aggregated): C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\news_daily_df.csv


In [3]:
# Connect to WRDS. This may prompt for credentials if no .pgpass file is configured.
db = wrds.Connection()

rp_tables = db.list_tables(library="rpna")
expected_rp_tables = [f"rpa_djpr_global_macro_{year}" for year in YEARS]
missing_rp_tables = [table for table in expected_rp_tables if table not in rp_tables]

if missing_rp_tables:
    raise RuntimeError(f"Missing RavenPack tables: {missing_rp_tables}")

print("WRDS connection ready.")
print("RavenPack macro tables found:", expected_rp_tables)

Loading library list...


Done
WRDS connection ready.
RavenPack macro tables found: ['rpa_djpr_global_macro_2020', 'rpa_djpr_global_macro_2021', 'rpa_djpr_global_macro_2022', 'rpa_djpr_global_macro_2023', 'rpa_djpr_global_macro_2024', 'rpa_djpr_global_macro_2025', 'rpa_djpr_global_macro_2026']


## Schema check: does a full-text column exist?

RavenPack's WRDS feed is licensed as sentiment/metadata analytics, not full-article redistribution, so the tables used below are not expected to carry a full article body column. Confirm this empirically against the live schema rather than assuming it — column availability can vary by RavenPack product tier.

In [4]:
macro_table_schema_df = db.describe_table("rpna", f"rpa_djpr_global_macro_{max(YEARS)}")
source_list_schema_df = db.describe_table("rpna", "rpa_source_list")

print("Columns in rpa_djpr_global_macro_<year>:")
display(macro_table_schema_df)

print("Columns in rpa_source_list:")
display(source_list_schema_df)

# Flags any column name that looks like it could hold free text (headline/body/summary/etc.)
text_like_columns = macro_table_schema_df.loc[
    macro_table_schema_df["name"].str.contains("headline|title|text|body|summary|content", case=False, na=False)
]
print("Columns that look like they might hold article text or headlines:")
display(text_like_columns)

Approximately 11279632 rows in rpna.rpa_djpr_global_macro_2026.
Approximately 100472 rows in rpna.rpa_source_list.
Columns in rpa_djpr_global_macro_<year>:


,name,nullable,type,comment
0,rpa_date_utc,True,DATE,None
1,rpa_time_utc,True,TIME,None
2,timestamp_utc,True,TIMESTAMP,None
3,rp_story_id,True,VARCHAR(32),None
4,rp_entity_id,True,VARCHAR(6),None
5,entity_type,True,VARCHAR(4),None
6,entity_name,True,VARCHAR(400),None
7,country_code,True,VARCHAR(2),None
8,relevance,True,DOUBLE PRECISION,None
9,event_sentiment_score,True,DOUBLE PRECISION,None


Columns in rpa_source_list:


,name,nullable,type,comment
0,rp_entity_id,True,VARCHAR(6),None
1,entity_type,True,VARCHAR(4),None
2,data_type,True,VARCHAR(20),None
3,data_value,True,VARCHAR(400),None
4,range_start,True,DATE,None
5,range_end,True,DATE,None


Columns that look like they might hold article text or headlines:


,name,nullable,type,comment
32,event_text,True,VARCHAR(400),None
51,headline,True,VARCHAR(4000),None


### Peek at actual `headline` / `event_text` values

Small, unfiltered-by-source sample so you can see real content and typical string lengths before deciding whether/how to bring these columns into the silver table. This output is not saved to disk — it is for interactive inspection only.

In [5]:
# Do not query or display licensed article text or headlines.
# The current deliverable uses RavenPack's structured sentiment fields only.
print("Raw headline/event_text inspection skipped by design.")

Raw headline/event_text inspection skipped by design.


## 1. Build the institutional source universe

Rank-1, non-blog sources only, same filter as `Basic_EDA_Analysis.ipynb`.

In [6]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


source_query = """
    SELECT rp_entity_id, data_type, data_value
    FROM rpna.rpa_source_list
    WHERE data_type IN ('ENTITY_NAME', 'PUBLICATION_TYPE', 'SOURCE_RANK')
"""

raw_source_attributes = db.raw_sql(source_query)

source_attribute_dupes = raw_source_attributes.duplicated(["rp_entity_id", "data_type"]).sum()
if source_attribute_dupes:
    raise ValueError(f"Source metadata has {source_attribute_dupes:,} duplicate entity/data_type rows; resolve before pivoting.")

sources_wide = raw_source_attributes.pivot(
    index="rp_entity_id", columns="data_type", values="data_value"
).reset_index()
sources_wide.columns.name = None

sources_df = sources_wide.rename(columns={
    "ENTITY_NAME": "source_name",
    "PUBLICATION_TYPE": "source_type",
    "SOURCE_RANK": "source_rank",
})

sources_df["source_rank"] = pd.to_numeric(sources_df["source_rank"], errors="coerce")
sources_df["is_rank1_non_blog"] = (
    sources_df["source_rank"].eq(1)
    & sources_df["source_type"].notna()
    & sources_df["source_type"].ne("BLOG")
)

institutional_sources_df = sources_df.loc[sources_df["is_rank1_non_blog"]].copy()
valid_source_ids = institutional_sources_df["rp_entity_id"].dropna().astype(str).tolist()
source_id_sql = sql_string_list(valid_source_ids)

print(f"All RavenPack source entities: {len(sources_df):,}")
print(f"Rank-1 non-blog institutional sources: {len(institutional_sources_df):,}")
display(institutional_sources_df.head())

All RavenPack source entities: 25,997
Rank-1 non-blog institutional sources: 394


,rp_entity_id,source_name,source_type,source_rank,is_rank1_non_blog
98,00F402,Independent News Ireland,NEWS,1,True
100,01032F,United Kingdom Parliament,NEWS,1,True
110,011E22,The Auto Channel,NEWS,1,True
170,01BDCE,The Mercury News,NEWS,1,True
199,01EFD4,Pressetext News,NEWS,1,True


## 2. Extract & union core filtered RavenPack events, 2020–2025 (Silver)

Row-level macro events from `rpna.rpa_djpr_global_macro_<year>`, filtered to relevance/event_relevance ≥ 90 and rank-1 non-blog sources, with the after-hours (4:00 PM ET) lookahead cutoff computed directly in SQL as `signal_calendar_date`. One year at a time, then unioned into a single table.

Includes `headline` and `event_text` so the NLP/LLM lead can run our own LLM sentiment scoring on the same text RavenPack scored, then compare our LLM-derived scores against `event_sentiment_score` (RavenPack's built-in score). These two columns are real article text, so this reinforces why the silver table stays under the gitignored `data/raw/` path.

In [7]:
ET_TIMESTAMP_SQL = "((timestamp_utc AT TIME ZONE 'UTC') AT TIME ZONE 'America/New_York')"


def fetch_core_events_for_year(year):
    table_name = f"rpna.rpa_djpr_global_macro_{year}"
    query = f"""
        SELECT
            rp_story_id,
            timestamp_utc,
            CASE
                WHEN {ET_TIMESTAMP_SQL}::time < TIME '{MARKET_CLOSE_ET}'
                    THEN {ET_TIMESTAMP_SQL}::date
                ELSE ({ET_TIMESTAMP_SQL}::date + INTERVAL '1 day')::date
            END AS signal_calendar_date,
            relevance,
            event_relevance,
            rp_source_id,
            source_name,
            topic,
            "group" AS group_name,
            event_sentiment_score,
            headline,
            event_text
        FROM {table_name}
        WHERE rpa_date_utc BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
          AND relevance >= {RELEVANCE_MIN}
          AND event_relevance >= {EVENT_RELEVANCE_MIN}
          AND rp_source_id IN ({source_id_sql})
          AND timestamp_utc IS NOT NULL
          AND event_sentiment_score IS NOT NULL
    """
    df = db.raw_sql(query)
    df["source_year"] = year
    return df


core_event_parts = []
for year in YEARS:
    print(f"Extracting core RavenPack macro events for {year}...")
    core_event_parts.append(fetch_core_events_for_year(year))

news_events_df = pd.concat(core_event_parts, ignore_index=True)

news_events_df["timestamp_utc"] = pd.to_datetime(news_events_df["timestamp_utc"])
news_events_df["signal_calendar_date"] = pd.to_datetime(news_events_df["signal_calendar_date"])
news_events_df["relevance"] = pd.to_numeric(news_events_df["relevance"], errors="coerce")
news_events_df["event_relevance"] = pd.to_numeric(news_events_df["event_relevance"], errors="coerce")
news_events_df["event_sentiment_score"] = pd.to_numeric(news_events_df["event_sentiment_score"], errors="coerce")

news_events_df = news_events_df.drop_duplicates(subset=["rp_story_id", "timestamp_utc", "rp_source_id"])

print(f"Core filtered event rows (2020-2025, unioned): {len(news_events_df):,}")
print(f"Distinct stories: {news_events_df['rp_story_id'].nunique():,}")
print(f"Rows with non-null headline: {news_events_df['headline'].notna().sum():,}")
print(f"Rows with non-null event_text: {news_events_df['event_text'].notna().sum():,}")
display(news_events_df.head())

Extracting core RavenPack macro events for 2020...


Extracting core RavenPack macro events for 2021...


Extracting core RavenPack macro events for 2022...


Extracting core RavenPack macro events for 2023...


Extracting core RavenPack macro events for 2024...


Extracting core RavenPack macro events for 2025...


Extracting core RavenPack macro events for 2026...


Core filtered event rows (2020-2025, unioned): 439,743
Distinct stories: 439,743
Rows with non-null headline: 439,743
Rows with non-null event_text: 439,743


,rp_story_id,timestamp_utc,signal_calendar_date,relevance,event_relevance,rp_source_id,source_name,topic,group_name,event_sentiment_score,headline,event_text,source_year
0,F54726F720E94F4960B95BF30459D5C3,2020-01-01 00:00:03.144,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.32,S Korea Dec Exports -5.2% On Year At $45.72B; ...,S Korea Dec Exports -5.2% On Year At $45.72B; ...,2020
1,F7B10F0B144B276261C543BB6BAE0EA8,2020-01-01 00:00:03.151,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,0.36,S Korea Dec Imports -0.7% On Year At $43.70B; ...,S Korea Dec Imports -0.7% On Year At $43.70B; ...,2020
2,EFAF78E8A157341DB2F18D9A0B0E0EA1,2020-01-01 00:00:03.159,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.74,S Korea Dec Trade Surplus $2.02B ; Forecast Su...,S Korea Dec Trade Surplus $2.02B ; Forecast Su...,2020
3,571254414CEBE531400952F385E5396B,2020-01-01 00:02:56.179,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.7,S Korea 2019 Exports -10.3% On Year Vs +5.4% i...,S Korea 2019 Exports -10.3% On Year,2020
4,0A92AFFCBD009E1902E7CC1AEDBF24EE,2020-01-01 00:57:00.485,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.49,South Korea's Exports Decline Moderates Signif...,South Korea's Exports Decline Moderates Signif...,2020


In [8]:
# Licensed WRDS data — write only to the gitignored raw/ folder, never to a tracked path.
news_events_df.to_csv(CORE_EVENTS_CSV, index=False)
print(f"Saved silver (row-level) table to: {CORE_EVENTS_CSV}")
print("This path is under data/raw/, which is .gitignore'd - do not force-add it.")

Saved silver (row-level) table to: C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\ravenpack_core_events_2020_2026.csv
This path is under data_collection/raw/, which is .gitignore'd - do not force-add it.


## 3. Infer the RavenPack sentiment scale

Computed directly from the unioned silver table (no need to re-query WRDS), same logic as `Basic_EDA_Analysis.ipynb`.

In [9]:
score_sample = news_events_df["event_sentiment_score"].dropna()

if score_sample.empty:
    raise RuntimeError("No event_sentiment_score values found in the unioned core events table.")

score_min = float(score_sample.min())
score_max = float(score_sample.max())

if score_min >= -1.5 and score_max <= 1.5:
    SENTIMENT_NEGATIVE_MAX = -0.05
    SENTIMENT_POSITIVE_MIN = 0.05
    SENTIMENT_SCALE_NOTE = "approx. -1 to 1 scale"
elif score_min >= 0 and score_max <= 100:
    SENTIMENT_NEGATIVE_MAX = 50.0
    SENTIMENT_POSITIVE_MIN = 50.0
    SENTIMENT_SCALE_NOTE = "approx. 0 to 100 scale"
else:
    SENTIMENT_NEGATIVE_MAX = -1.0
    SENTIMENT_POSITIVE_MIN = 1.0
    SENTIMENT_SCALE_NOTE = "wide signed scale fallback"

print({
    "rows": len(score_sample),
    "min": score_min,
    "mean": float(score_sample.mean()),
    "max": score_max,
    "scale_note": SENTIMENT_SCALE_NOTE,
    "negative_if_less_than": SENTIMENT_NEGATIVE_MAX,
    "positive_if_greater_than": SENTIMENT_POSITIVE_MIN,
})

{'rows': 439743, 'min': -1.0, 'mean': 0.046036093809338645, 'max': 1.0, 'scale_note': 'approx. -1 to 1 scale', 'negative_if_less_than': -0.05, 'positive_if_greater_than': 0.05}


## 4. Build the daily aggregated summary table (Gold): `news_daily_df`

Two-stage aggregation, same approach as `Basic_EDA_Analysis.ipynb`:

1. Aggregate the silver table by `signal_calendar_date` (the 4:00 PM ET cutoff already applied).
2. Map each calendar date to the next NYSE trading session using `pandas_market_calendars` (no CRSP dependency needed in this notebook), then re-aggregate to `session_date` so news from non-trading days (weekends/holidays) rolls forward correctly.

In [10]:
import pandas_market_calendars as mcal

news_events_df["is_positive"] = news_events_df["event_sentiment_score"] > SENTIMENT_POSITIVE_MIN
news_events_df["is_negative"] = news_events_df["event_sentiment_score"] < SENTIMENT_NEGATIVE_MAX
news_events_df["is_neutral"] = ~news_events_df["is_positive"] & ~news_events_df["is_negative"]

calendar_daily_df = (
    news_events_df
    .groupby("signal_calendar_date")
    .agg(
        event_record_count=("rp_story_id", "size"),
        unique_story_count=("rp_story_id", "nunique"),
        positive_event_count=("is_positive", "sum"),
        negative_event_count=("is_negative", "sum"),
        neutral_event_count=("is_neutral", "sum"),
        unique_source_count=("rp_source_id", "nunique"),
        sentiment_sum=("event_sentiment_score", "sum"),
    )
    .reset_index()
)

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=START_DATE, end_date=(pd.Timestamp(END_DATE) + pd.Timedelta(days=7)))
trading_sessions = pd.Series(pd.to_datetime(schedule.index)).sort_values()


def map_to_next_trading_session(dates, sessions):
    session_values = sessions.to_numpy(dtype="datetime64[ns]")
    target_values = pd.to_datetime(dates).to_numpy(dtype="datetime64[ns]")
    positions = np.searchsorted(session_values, target_values, side="left")
    mapped = np.full(len(target_values), np.datetime64("NaT"), dtype="datetime64[ns]")
    valid = positions < len(session_values)
    mapped[valid] = session_values[positions[valid]]
    return pd.to_datetime(mapped)


calendar_daily_df["session_date"] = map_to_next_trading_session(
    calendar_daily_df["signal_calendar_date"], trading_sessions
)
calendar_daily_df = calendar_daily_df.dropna(subset=["session_date"])

news_daily_df = (
    calendar_daily_df
    .groupby("session_date", as_index=False)
    .agg(
        event_record_count=("event_record_count", "sum"),
        unique_story_count=("unique_story_count", "sum"),
        positive_event_count=("positive_event_count", "sum"),
        negative_event_count=("negative_event_count", "sum"),
        neutral_event_count=("neutral_event_count", "sum"),
        unique_source_count=("unique_source_count", "max"),
        sentiment_sum=("sentiment_sum", "sum"),
    )
)

# Recompute source breadth from event rows after session mapping. A max across
# calendar dates can undercount sources when a weekend/holiday rolls forward.
event_session_sources = news_events_df[["signal_calendar_date", "rp_source_id"]].copy()
event_session_sources["session_date"] = map_to_next_trading_session(
    event_session_sources["signal_calendar_date"], trading_sessions
)
source_counts_by_session = (
    event_session_sources.dropna(subset=["session_date"])
    .groupby("session_date")["rp_source_id"]
    .nunique()
)
news_daily_df["unique_source_count"] = (
    news_daily_df["session_date"].map(source_counts_by_session).fillna(0).astype(int)
)

news_daily_df["mean_event_sentiment_score"] = (
    news_daily_df["sentiment_sum"] / news_daily_df["event_record_count"]
)
news_daily_df = news_daily_df.drop(columns=["sentiment_sum"])

for label in ["positive", "negative", "neutral"]:
    news_daily_df[f"{label}_event_share"] = (
        news_daily_df[f"{label}_event_count"] / news_daily_df["event_record_count"]
    )

conditions = [
    news_daily_df["mean_event_sentiment_score"] > SENTIMENT_POSITIVE_MIN,
    news_daily_df["mean_event_sentiment_score"] < SENTIMENT_NEGATIVE_MAX,
]
news_daily_df["sentiment_bucket"] = np.select(conditions, ["positive", "negative"], default="neutral")
news_daily_df = news_daily_df.sort_values("session_date").reset_index(drop=True)

print(f"Trading-session news rows: {len(news_daily_df):,}")
display(news_daily_df.head())

Trading-session news rows: 1,632


,session_date,event_record_count,unique_story_count,positive_event_count,negative_event_count,neutral_event_count,unique_source_count,mean_event_sentiment_score,positive_event_share,negative_event_share,neutral_event_share,sentiment_bucket
0,2020-01-02,182,182,61,77,44,5,-0.036264,0.335165,0.423077,0.241758,neutral
1,2020-01-03,264,264,116,98,50,4,0.018674,0.439394,0.371212,0.189394,neutral
2,2020-01-06,250,250,112,65,73,7,0.0636,0.448,0.26,0.292,positive
3,2020-01-07,296,296,99,116,81,9,-0.018378,0.334459,0.391892,0.273649,neutral
4,2020-01-08,328,328,92,197,39,9,-0.202896,0.280488,0.60061,0.118902,negative


## 5. Validation checks

In [11]:
assert (news_daily_df["unique_story_count"] <= news_daily_df["event_record_count"]).all(), (
    "unique_story_count should not exceed event_record_count"
)

share_sum = news_daily_df[["positive_event_share", "negative_event_share", "neutral_event_share"]].sum(axis=1)
assert np.allclose(share_sum, 1.0, atol=1e-6), "Sentiment shares do not sum to 1.0"

duplicate_sessions = news_daily_df["session_date"].duplicated().sum()
assert duplicate_sessions == 0, f"Duplicate session_date rows found: {duplicate_sessions}"

print("Validation checks passed.")
print(f"Calendar date range: {news_events_df['signal_calendar_date'].min().date()} to {news_events_df['signal_calendar_date'].max().date()}")
print(f"Session date range:  {news_daily_df['session_date'].min().date()} to {news_daily_df['session_date'].max().date()}")

# Spot-check (visual only): after-hours (>= 4pm ET) articles should roll to the next trading session,
# not the same day - this is the lookahead-bias guard called out in CLAUDE.md.
after_hours_mask = (
    news_events_df["timestamp_utc"].dt.tz_localize("UTC").dt.tz_convert("America/New_York").dt.time
    >= pd.Timestamp(MARKET_CLOSE_ET).time()
)
after_hours_spot_check = news_events_df.loc[
    after_hours_mask, ["timestamp_utc", "signal_calendar_date", "rp_story_id"]
].head(5).copy()
after_hours_spot_check["mapped_session_date"] = map_to_next_trading_session(
    after_hours_spot_check["signal_calendar_date"], trading_sessions
)
display(after_hours_spot_check)

Validation checks passed.
Calendar date range: 2020-01-01 to 2026-07-01
Session date range:  2020-01-02 to 2026-07-01


,timestamp_utc,signal_calendar_date,rp_story_id,mapped_session_date
0,2020-01-01 00:00:03.144,2020-01-01,F54726F720E94F4960B95BF30459D5C3,2020-01-02
1,2020-01-01 00:00:03.151,2020-01-01,F7B10F0B144B276261C543BB6BAE0EA8,2020-01-02
2,2020-01-01 00:00:03.159,2020-01-01,EFAF78E8A157341DB2F18D9A0B0E0EA1,2020-01-02
3,2020-01-01 00:02:56.179,2020-01-01,571254414CEBE531400952F385E5396B,2020-01-02
4,2020-01-01 00:57:00.485,2020-01-01,0A92AFFCBD009E1902E7CC1AEDBF24EE,2020-01-02


## 6. Save the gold summary table and print a final run summary

`news_daily_df.csv` is aggregated/derived, not raw WRDS records, so it is safe to commit per the licensing constraint in `CLAUDE.md`.

In [12]:
news_daily_df.to_csv(NEWS_DAILY_CSV, index=False)

print("Extraction complete.")
print(f"Silver (row-level, gitignored):     {CORE_EVENTS_CSV}  [{len(news_events_df):,} rows]")
print(f"Gold (daily summary, committed):     {NEWS_DAILY_CSV}  [{len(news_daily_df):,} rows]")
print(f"Rank-1 non-blog sources used:        {len(institutional_sources_df):,}")
print(f"Sentiment scale detected:            {SENTIMENT_SCALE_NOTE}")

Extraction complete.
Silver (row-level, gitignored):     C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\ravenpack_core_events_2020_2026.csv  [439,743 rows]
Gold (daily summary, committed):     C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\news_daily_df.csv  [1,632 rows]
Rank-1 non-blog sources used:        394
Sentiment scale detected:            approx. -1 to 1 scale
